# Notebook 06 — Train CNN Networks

Trains two ReLU-only CNN classifiers for concolic robustness experiments.

| Model | Dataset | Architecture | Est. Time (CPU) |
|---|---|---|---|
| `lenet5_mnist` | MNIST | Conv6→Conv16→FC120→FC84→10 | ~15 min |
| `smallcnn_cifar` | CIFAR-10 | Conv32→Conv64→Conv128→FC256→10 | ~20 min |

**Why SmallCNN instead of VGG-11?**  
VGG-11 (9.7M params) takes ~6 hours on CPU. SmallCNN (~540K params) achieves
~72–75% on CIFAR-10 in ~20 minutes while providing three conv layers with
pooling — architecturally meaningful for activation-region analysis.
VGG-11 can still be trained on Kaggle GPU if desired (change `SmallCNN` → `VGG11ReLU`).

**Design principle:** All networks use ONLY ReLU — no BatchNorm, no Dropout.
This preserves piecewise-linear structure for activation-region analysis.

**Outputs:**
```
models/lenet5_mnist.pt
models/smallcnn_cifar.pt
results/cnn_training_summary.json
```

In [1]:
import subprocess, sys
for pkg in ['torch', 'torchvision', 'tqdm']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'], check=False)
import torch
print(f'PyTorch: {torch.__version__}')

PyTorch: 2.12.0+cpu


In [2]:
import sys, os, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from pathlib import Path
from tqdm import tqdm

REPO_ROOT = Path(os.getcwd())
sys.path.insert(0, str(REPO_ROOT))

from utils.cnn_definitions import LeNet5, SmallCNN, save_cnn, load_cnn
from utils.metrics import save_results

MODELS_DIR  = REPO_ROOT / 'models'
RESULTS_DIR = REPO_ROOT / 'results'
DATA_DIR    = REPO_ROOT / 'data'
for d in [MODELS_DIR, RESULTS_DIR, DATA_DIR]:
    d.mkdir(exist_ok=True)

DEVICE = 'cpu'
SEED   = 42
torch.manual_seed(SEED); np.random.seed(SEED)
torch.set_num_threads(os.cpu_count() or 2)
print(f'CPU threads: {os.cpu_count()}')

CPU threads: 8


## 1 — Data loaders

In [3]:
# MNIST — pad to 32×32 for LeNet5
mnist_tf = transforms.Compose([
    transforms.Pad(2),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
mnist_train = datasets.MNIST(DATA_DIR, train=True,  download=True, transform=mnist_tf)
mnist_test  = datasets.MNIST(DATA_DIR, train=False, download=True, transform=mnist_tf)
mnist_tl    = DataLoader(mnist_train, batch_size=128, shuffle=True,  num_workers=0)
mnist_vl    = DataLoader(mnist_test,  batch_size=256, shuffle=False, num_workers=0)
print(f'MNIST    train={len(mnist_train):,}  test={len(mnist_test):,}')

# CIFAR-10
cifar_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),          # light augmentation
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])
cifar_tf_val = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])
cifar_train = datasets.CIFAR10(DATA_DIR, train=True,  download=True, transform=cifar_tf)
cifar_test  = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=cifar_tf_val)
cifar_tl    = DataLoader(cifar_train, batch_size=128, shuffle=True,  num_workers=0)
cifar_vl    = DataLoader(cifar_test,  batch_size=256, shuffle=False, num_workers=0)
print(f'CIFAR-10 train={len(cifar_train):,}  test={len(cifar_test):,}')

MNIST    train=60,000  test=10,000
CIFAR-10 train=50,000  test=10,000


## 2 — Training function

In [4]:
def train_cnn(model, train_loader, val_loader,
              epochs=25, lr=1e-3, weight_decay=1e-4, model_name=''):
    model     = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history   = {'train_loss': [], 'train_acc': [], 'test_acc': []}
    best_acc  = 0.0
    t0        = time.time()

    bar = tqdm(range(1, epochs + 1), desc=model_name, unit='ep')
    for epoch in bar:
        model.train()
        total_loss = correct = total = 0
        for X, y in train_loader:
            optimizer.zero_grad()
            out  = model(X)          # CNN — no flattening
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * y.size(0)
            correct    += (out.argmax(1) == y).sum().item()
            total      += y.size(0)
        scheduler.step()

        model.eval()
        c_t = t_t = 0
        with torch.no_grad():
            for X, y in val_loader:
                c_t += (model(X).argmax(1) == y).sum().item()
                t_t += y.size(0)
        te_acc = c_t / t_t
        best_acc = max(best_acc, te_acc)

        history['train_loss'].append(round(total_loss/total, 4))
        history['train_acc'].append(round(correct/total,    4))
        history['test_acc'].append(round(te_acc,           4))

        bar.set_postfix(
            loss=f'{total_loss/total:.3f}',
            train=f'{correct/total*100:.1f}%',
            test=f'{te_acc*100:.1f}%',
            best=f'{best_acc*100:.1f}%',
            t=f'{time.time()-t0:.0f}s'
        )

    model.eval()
    print(f'  Best test acc: {best_acc*100:.2f}%  |  '
          f'Total time: {time.time()-t0:.0f}s')
    return history

## 3 — Train LeNet-5 on MNIST
**Expected:** ~99% accuracy in ~15 min

In [5]:
lenet = LeNet5(input_channels=1, num_classes=10, input_size=32)
print(lenet)
print(f'Params: {sum(p.numel() for p in lenet.parameters()):,}\n')

lenet_history = train_cnn(
    lenet, mnist_tl, mnist_vl,
    epochs=20, lr=1e-3, model_name='LeNet5-MNIST',
)

save_cnn(lenet, str(MODELS_DIR / 'lenet5_mnist.pt'), metadata={
    'dataset'      : 'MNIST',
    'input_size'   : 32,
    'best_test_acc': max(lenet_history['test_acc']),
    'architecture' : 'LeNet5-ReLU-noBN',
    'n_params'     : sum(p.numel() for p in lenet.parameters()),
})

LeNet5(in=1ch, classes=10, flat=576, params=82,826)
Params: 82,826



LeNet5-MNIST: 100%|███████████| 20/20 [08:37<00:00, 25.88s/ep, best=99.2%, loss=0.005, t=518s, test=99.1%, train=99.9%]


  Best test acc: 99.18%  |  Total time: 518s
  Saved → D:\concolic_exploration\models\lenet5_mnist.pt


## 4 — Train SmallCNN on CIFAR-10
**Expected:** ~72–75% accuracy in ~20 min  

> **To use VGG-11 instead** (needs Kaggle GPU):  
> Replace `SmallCNN(...)` with `VGG11ReLU(...)` and change filename to `vgg11_cifar.pt`  
> Expected: ~83% accuracy, ~40 min on GPU

In [6]:
smallcnn = SmallCNN(input_channels=3, num_classes=10, input_size=32)
print(smallcnn)
print(f'Params: {sum(p.numel() for p in smallcnn.parameters()):,}\n')

smallcnn_history = train_cnn(
    smallcnn, cifar_tl, cifar_vl,
    epochs=30, lr=1e-3, weight_decay=1e-4, model_name='SmallCNN-CIFAR10',
)

save_cnn(smallcnn, str(MODELS_DIR / 'smallcnn_cifar.pt'), metadata={
    'dataset'      : 'CIFAR-10',
    'input_size'   : 32,
    'best_test_acc': max(smallcnn_history['test_acc']),
    'architecture' : 'SmallCNN-3conv-ReLU-noBN',
    'n_params'     : sum(p.numel() for p in smallcnn.parameters()),
})

SmallCNN(in=3ch, classes=10, flat=2048, params=620,362)
Params: 620,362



SmallCNN-CIFAR10: 100%|██████| 30/30 [33:50<00:00, 67.68s/ep, best=80.9%, loss=0.022, t=2030s, test=80.7%, train=99.8%]

  Best test acc: 80.90%  |  Total time: 2030s
  Saved → D:\concolic_exploration\models\smallcnn_cifar.pt


## 5 — Summary & verify

In [7]:
summary = {
    'lenet5_mnist': {
        'best_test_acc': max(lenet_history['test_acc']),
        'history'      : lenet_history,
        'architecture' : 'LeNet5-ReLU-noBN',
        'n_params'     : sum(p.numel() for p in lenet.parameters()),
        'dataset'      : 'MNIST (32×32 padded)',
    },
    'smallcnn_cifar': {
        'best_test_acc': max(smallcnn_history['test_acc']),
        'history'      : smallcnn_history,
        'architecture' : 'SmallCNN-3conv-ReLU-noBN',
        'n_params'     : sum(p.numel() for p in smallcnn.parameters()),
        'dataset'      : 'CIFAR-10',
    },
}
save_results(summary, str(RESULTS_DIR / 'cnn_training_summary.json'))

# verify reload
print('Verifying saved models...')
for name, path in [('lenet5_mnist',   MODELS_DIR / 'lenet5_mnist.pt'),
                   ('smallcnn_cifar', MODELS_DIR / 'smallcnn_cifar.pt')]:
    m   = load_cnn(str(path))
    out = m(torch.randn(1, *m.input_shape))
    assert out.shape == (1, 10)
    print(f'  {name:<22} shape={list(out.shape)}  ✓')

print()
print('═'*60)
print('  CNN TRAINING COMPLETE')
print('═'*60)
for name, info in summary.items():
    print(f"  {name:<22} "
          f"best_acc={info['best_test_acc']*100:.2f}%  "
          f"params={info['n_params']:,}")
print()
print('  Next step → run 07_certified_baselines.ipynb')

  Saved results → D:\concolic_exploration\results\cnn_training_summary.json
Verifying saved models...
  Loaded ← D:\concolic_exploration\models\lenet5_mnist.pt  |  metadata: {'dataset': 'MNIST', 'input_size': 32, 'best_test_acc': 0.9918, 'architecture': 'LeNet5-ReLU-noBN', 'n_params': 82826}
  lenet5_mnist           shape=[1, 10]  ✓
  Loaded ← D:\concolic_exploration\models\smallcnn_cifar.pt  |  metadata: {'dataset': 'CIFAR-10', 'input_size': 32, 'best_test_acc': 0.809, 'architecture': 'SmallCNN-3conv-ReLU-noBN', 'n_params': 620362}
  smallcnn_cifar         shape=[1, 10]  ✓

════════════════════════════════════════════════════════════
  CNN TRAINING COMPLETE
════════════════════════════════════════════════════════════
  lenet5_mnist           best_acc=99.18%  params=82,826
  smallcnn_cifar         best_acc=80.90%  params=620,362

  Next step → run 07_certified_baselines.ipynb
